# Chapter 16 Computational Lab
## Discrete-Time Markov Chains

This notebook accompanies Chapter 16 of *Probability Theory with Python and AI*.

A Markov chain is the first major model in the book in which dependence through time is central. The future need not be independent of the past. Instead,

$$
\boxed{
\text{past}
\longrightarrow
X_n
\longrightarrow
\text{future}
}
$$

means that once the present state is known, the more distant past gives no additional information about future transitions.

The chapter develops the theory from conditional expectation and transition matrices through hitting times, recurrence, stationarity, reversibility, periodicity, residue-class limits and ergodic averages.

### Learning goals

By the end of the lab you should be able to:

1. distinguish a stochastic process from a single random variable;
2. state the time-homogeneous Markov property correctly;
3. explain why Markov dependence is not independence;
4. use the natural filtration and Markov operator;
5. interpret $(P^n)_{ij}$ as an $n$-step transition probability;
6. compute path probabilities and apply Chapman--Kolmogorov;
7. formulate hitting and return times;
8. recognize hitting times as stopping times;
9. use first-step equations for hitting probabilities and expected hitting times;
10. understand the role of the strong Markov property;
11. determine communication classes and irreducibility;
12. distinguish recurrence, transience, positive recurrence and null recurrence;
13. use the recurrence criterion $\sum_n(P^n)_{ii}$;
14. compute stationary distributions;
15. interpret Kac's return-time formula;
16. use detailed balance and reversibility;
17. compute periods using return times or the directed graph;
18. construct cyclic classes;
19. understand why each cyclic class has stationary mass $1/d$;
20. apply the finite irreducible aperiodic convergence theorem;
21. compute all residue-class subsequential limits of a periodic chain;
22. distinguish finite-state and countably infinite limit manipulations;
23. compute Cesàro limits and explain why periodic oscillation disappears under averaging;
24. understand the sample-path ergodic theorem;
25. solve gambler's-ruin probabilities and expected durations;
26. classify one-dimensional simple random walks as recurrent or transient;
27. audit AI-generated Markov-chain claims.

> **Key distinction.** A stationary distribution can exist and be unique even when $P^n$ does not converge. Periodicity is the obstruction.


## 0. Setup

The computational functions below mirror the chapter's finite-state Python laboratory.


In [ ]:
from collections import deque
from math import gcd
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import HTML, Math, Markdown, clear_output, display

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass


def is_stochastic(P, tol=1e-12):
    P = np.asarray(P, dtype=float)

    return (
        P.ndim == 2
        and P.shape[0] == P.shape[1]
        and np.all(P >= -tol)
        and np.allclose(P.sum(axis=1), 1, atol=tol)
    )


def stationary_distribution(P):
    """Stationary row vector pi satisfying pi @ P = pi."""
    P = np.asarray(P, dtype=float)
    m = P.shape[0]

    A = P.T - np.eye(m)
    b = np.zeros(m)

    A[-1, :] = 1.0
    b[-1] = 1.0

    return np.linalg.solve(A, b)


def matrix_power(P, n):
    return np.linalg.matrix_power(
        np.asarray(P, dtype=float),
        int(n),
    )


def law_after_n(mu, P, n):
    return np.asarray(mu, dtype=float) @ matrix_power(P, n)


def path_probability(mu, P, path):
    mu = np.asarray(mu, dtype=float)
    P = np.asarray(P, dtype=float)
    path = list(path)

    value = mu[path[0]]

    for a, b in zip(path[:-1], path[1:]):
        value *= P[a, b]

    return float(value)


def directed_reachable(P, start, tol=1e-14):
    P = np.asarray(P, dtype=float)
    m = P.shape[0]

    seen = {start}
    q = deque([start])

    while q:
        i = q.popleft()

        for j in range(m):
            if P[i, j] > tol and j not in seen:
                seen.add(j)
                q.append(j)

    return seen


def is_irreducible(P, tol=1e-14):
    P = np.asarray(P, dtype=float)
    m = P.shape[0]

    return all(
        len(directed_reachable(P, i, tol)) == m
        for i in range(m)
    )


def communication_classes(P, tol=1e-14):
    P = np.asarray(P, dtype=float)
    m = P.shape[0]

    reach = [
        directed_reachable(P, i, tol)
        for i in range(m)
    ]

    unused = set(range(m))
    classes = []

    while unused:
        i = min(unused)
        cls = {
            j for j in unused
            if j in reach[i] and i in reach[j]
        }

        classes.append(sorted(cls))
        unused -= cls

    return classes


def period_irreducible(P, tol=1e-14):
    """Period using the graph gcd formula from one reference state."""
    P = np.asarray(P, dtype=float)
    m = P.shape[0]

    if not is_irreducible(P, tol):
        raise ValueError("The matrix must be irreducible.")

    adj = [
        [j for j in range(m) if P[i, j] > tol]
        for i in range(m)
    ]

    dist = [-1]*m
    dist[0] = 0
    q = deque([0])

    while q:
        i = q.popleft()

        for j in adj[i]:
            if dist[j] == -1:
                dist[j] = dist[i] + 1
                q.append(j)

    d = 0

    for i in range(m):
        for j in adj[i]:
            discrepancy = dist[i] + 1 - dist[j]

            if discrepancy != 0:
                d = gcd(d, abs(discrepancy))

    return d


def cyclic_classes(P, tol=1e-14):
    """Return cyclic classes for a finite irreducible chain."""
    P = np.asarray(P, dtype=float)
    d = period_irreducible(P, tol)
    m = P.shape[0]

    adj = [
        [j for j in range(m) if P[i, j] > tol]
        for i in range(m)
    ]

    residue = [None]*m
    residue[0] = 0
    q = deque([0])

    while q:
        i = q.popleft()

        for j in adj[i]:
            r = (residue[i] + 1) % d

            if residue[j] is None:
                residue[j] = r
                q.append(j)

    return [
        [i for i, r in enumerate(residue) if r == a]
        for a in range(d)
    ]


def cesaro_average(P, N):
    P = np.asarray(P, dtype=float)
    m = P.shape[0]

    total = np.zeros((m, m))
    current = np.eye(m)

    for _ in range(N):
        total += current
        current = current @ P

    return total/N


def detailed_balance_error(P, pi):
    P = np.asarray(P, dtype=float)
    pi = np.asarray(pi, dtype=float)

    flows = pi[:, None]*P
    return float(np.max(np.abs(flows-flows.T)))


def hitting_probability(P, A, B):
    """h_i=P_i(T_A<T_B) for a finite chain."""
    P = np.asarray(P, dtype=float)
    m = P.shape[0]

    A = set(A)
    B = set(B)

    h = np.zeros(m)
    h[list(A)] = 1.0

    interior = [
        i for i in range(m)
        if i not in A and i not in B
    ]

    if not interior:
        return h

    Q = P[np.ix_(interior, interior)]
    rhs = P[np.ix_(interior, sorted(A))].sum(axis=1)

    h_inner = np.linalg.solve(
        np.eye(len(interior)) - Q,
        rhs,
    )

    h[interior] = h_inner
    return h


def expected_hitting_time(P, A):
    """Expected time to hit A, assuming the finite linear system is solvable."""
    P = np.asarray(P, dtype=float)
    m = P.shape[0]

    A = set(A)

    t = np.zeros(m)

    interior = [
        i for i in range(m)
        if i not in A
    ]

    if not interior:
        return t

    Q = P[np.ix_(interior, interior)]

    t_inner = np.linalg.solve(
        np.eye(len(interior)) - Q,
        np.ones(len(interior)),
    )

    t[interior] = t_inner
    return t


def simulate_chain(P, x0, N, seed=2026):
    P = np.asarray(P, dtype=float)
    rng = np.random.default_rng(seed)

    states = np.empty(N, dtype=int)
    x = int(x0)

    for n in range(N):
        states[n] = x
        x = rng.choice(P.shape[0], p=P[x])

    return states


def gambler_ruin_probability(i, N, p):
    q = 1-p

    if p == 0:
        return 0.0

    if p == 1:
        return 1.0

    if abs(p-0.5) < 1e-14:
        return i/N

    r = q/p

    return (
        1-r**i
    ) / (
        1-r**N
    )


def gambler_ruin_duration(i, N, p):
    q = 1-p

    if p == 0:
        return float(i)

    if p == 1:
        return float(N-i)

    if abs(p-0.5) < 1e-14:
        return float(i*(N-i))

    r = q/p

    return (
        i/(q-p)
        -
        N/(q-p)
        *
        (1-r**i)/(1-r**N)
    )


def show_result(title, *latex_lines, note=None):
    display(HTML(
        f"<div style='border-left:5px solid;padding:8px 12px;margin:8px 0'>"
        f"<b>{title}</b></div>"
    ))

    for line in latex_lines:
        display(Math(line))

    if note:
        display(Markdown(note))


display(HTML(
    "<div style='padding:10px;border:1px solid'>"
    "<b>Setup complete.</b> Markov-chain tools are ready."
    "</div>"
))


## 1. From random variables to stochastic processes

A **discrete-time stochastic process** is a sequence

$$
X_0,X_1,X_2,\ldots
$$

of random variables on the same probability space.

For fixed $\omega$, the sequence

$$
X_0(\omega),X_1(\omega),X_2(\omega),\ldots
$$

is a trajectory or sample path.


### Time-homogeneous Markov property

A Markov chain with transition matrix

$$
P=(p_{ij})
$$

satisfies, for every positive-probability history,

$$
\boxed{
P(
X_{n+1}=j
\mid
X_0=i_0,\ldots,X_n=i_n
)
=
p_{i_nj}.
}
$$

The transition matrix is part of the model specification, including rows corresponding to states that might not be reached under one particular initial distribution.


### Markov dependence is not independence

The Markov property does **not** say that

$$
X_{n+1}
$$

is independent of

$$
X_n.
$$

Knowing the present state may strongly change the distribution of the next state.

The condition is instead:

$$
\boxed{
\text{given }X_n,
\text{ the earlier path supplies no additional one-step information.}
}
$$


In [ ]:
P_two = np.array([
    [0.8,0.2],
    [0.3,0.7],
])

display(Markdown(f"Stochastic matrix: **{is_stochastic(P_two)}**"))
display(Math(r"P(X_{n+1}=2\mid X_n=1)=0.2"))
display(Math(r"P(X_{n+1}=2\mid X_n=2)=0.7"))
display(Markdown(
    "The next-state law depends on the current state, so successive states are generally dependent."
))


## 2. Natural filtration and Markov operator

The natural filtration is

$$
\boxed{
\mathcal F_n
=
\sigma(X_0,\ldots,X_n).
}
$$

It represents all information revealed by the chain up to time $n$.


For bounded $g:S\to\mathbb R$, define the Markov operator

$$
\boxed{
(Pg)(i)
=
\sum_jp_{ij}g(j).
}
$$

The conditional-expectation form of the Markov property is

$$
\boxed{
\mathbb E[
g(X_{n+1})
\mid
\mathcal F_n
]
=
(Pg)(X_n).
}
$$

More generally,

$$
\boxed{
\mathbb E[
g(X_{n+k})
\mid
\mathcal F_n
]
=
(P^kg)(X_n).
}
$$


### Two-state forecast

Let

$$
P=
\begin{pmatrix}
0.8&0.2\\
0.3&0.7
\end{pmatrix},
$$

with

$$
g(1)=0,
\qquad
g(2)=10.
$$

Then

$$
(Pg)(1)=2,
\qquad
(Pg)(2)=7.
$$

The present state compresses the relevant predictive information from the whole past.


In [ ]:
g = np.array([0.0,10.0])
Pg = P_two @ g

display(Math(r"(Pg)(1)=" + f"{Pg[0]:.1f}"))
display(Math(r"(Pg)(2)=" + f"{Pg[1]:.1f}"))


## 3. Transition matrices and multi-step probabilities

For a finite state space,

$$
P=(p_{ij})
$$

is stochastic:

$$
p_{ij}\ge0,
\qquad
\sum_jp_{ij}=1.
$$

We use row vectors for probability distributions.


### Evolution of the distribution

If the initial law is

$$
\mu^{(0)},
$$

then

$$
\boxed{
\mu^{(n)}
=
\mu^{(0)}P^n.
}
$$

If the chain starts from state $i$,

$$
\boxed{
P_i(X_n=j)
=
(P^n)_{ij}.
}
$$


In [ ]:
evolve_n = widgets.IntSlider(
    value=5,
    min=0,
    max=30,
    description="n",
)
evolve_output = widgets.Output()


def update_evolution(*_):
    with evolve_output:
        clear_output(wait=True)

        n = evolve_n.value
        mu0 = np.array([1.0,0.0])
        mun = law_after_n(mu0,P_two,n)

        display(Markdown(
            f"$\\mu^{{({n})}}$ = **{np.round(mun,6)}**"
        ))


evolve_n.observe(update_evolution, names="value")
display(widgets.VBox([evolve_n,evolve_output]))
update_evolution()


### Probability of a path

For states $i_0,\ldots,i_n$,

$$
\boxed{
P(
X_0=i_0,\ldots,X_n=i_n
)
=
\mu^{(0)}_{i_0}
\prod_{r=0}^{n-1}
p_{i_ri_{r+1}}.
}
$$

This is the time-dependent analogue of multiplying conditional probabilities along a chain of events.


In [ ]:
mu0 = np.array([1.0,0.0])
path = [0,0,1,1]

prob = path_probability(
    mu0,
    P_two,
    path,
)

display(Math(
    r"P(1,1,2,2)="
    + f"{prob:.6f}"
))
display(Math(r"=1(0.8)(0.2)(0.7)"))


### Chapman--Kolmogorov

For non-negative integers $r,s$,

$$
\boxed{
(P^{r+s})_{ij}
=
\sum_k
(P^r)_{ik}
(P^s)_{kj}.
}
$$

This is matrix multiplication interpreted as conditioning on an intermediate state.


In [ ]:
r,s = 2,3

lhs = matrix_power(P_two,r+s)
rhs = matrix_power(P_two,r) @ matrix_power(P_two,s)

display(Markdown(
    f"Maximum Chapman--Kolmogorov discrepancy: **{np.max(np.abs(lhs-rhs)):.3e}**"
))


## 4. Hitting times, return times and stopping times

For $A\subseteq S$, define

$$
\boxed{
T_A
=
\inf\{n\ge0:X_n\in A\}.
}
$$

For one state $i$,

$$
T_i=T_{\{i\}},
$$

and the first positive return time is

$$
\boxed{
T_i^+
=
\inf\{n\ge1:X_n=i\}.
}
$$


A random time $\tau$ is a stopping time if

$$
\boxed{
\{\tau\le n\}
\in
\mathcal F_n
}
$$

for every $n$.

For a hitting time,

$$
\{T_A\le n\}
=
\bigcup_{r=0}^n
\{X_r\in A\},
$$

so hitting times are stopping times.


### Strong Markov property

At a stopping time $\tau$,

$$
\boxed{
\mathbb E[
\mathbf1_{\{\tau<\infty\}}
g(X_{\tau+k})
\mid
\mathcal F_\tau
]
=
\mathbf1_{\{\tau<\infty\}}
(P^kg)(X_\tau).
}
$$

Conditional on the state reached at the stopping time, the future evolves as a fresh chain started from that state.


## 5. First-step analysis

For disjoint target sets $A,B$, define

$$
h_i
=
P_i(T_A<T_B).
$$

Then

$$
h_i=1
\quad
(i\in A),
$$

$$
h_i=0
\quad
(i\in B),
$$

and for interior states,

$$
\boxed{
h_i
=
\sum_jp_{ij}h_j.
}
$$


For a target $A$, define

$$
m_i
=
\mathbb E_i[T_A].
$$

Then

$$
m_i=0
\quad
(i\in A),
$$

while for $i\notin A$,

$$
\boxed{
m_i
=
1+\sum_jp_{ij}m_j.
}
$$

These equations are the main computational bridge from Markov chains to linear algebra.


### A target with a possible delay

Consider states $\{0,1,2\}$, with $0$ and $2$ absorbing, and from state $1$ use probabilities

$$
0.2,\ 0.5,\ 0.3
$$

to states $0,1,2$.

Then

$$
h_1=0.5h_1+0.3,
$$

so

$$
\boxed{
h_1=0.6.
}
$$

Also,

$$
m_1=1+0.5m_1,
$$

so

$$
\boxed{
m_1=2.
}
$$


In [ ]:
P_delay = np.array([
    [1.0,0.0,0.0],
    [0.2,0.5,0.3],
    [0.0,0.0,1.0],
])

h = hitting_probability(
    P_delay,
    A={2},
    B={0},
)

m = expected_hitting_time(
    P_delay,
    A={0,2},
)

display(Math(r"h_1=" + f"{h[1]:.6f}"))
display(Math(r"m_1=" + f"{m[1]:.6f}"))


## 6. Communication and irreducibility

State $i$ **leads to** $j$, written

$$
i\to j,
$$

if

$$
(P^n)_{ij}>0
$$

for some $n\ge0$.

States communicate when

$$
\boxed{
i\leftrightarrow j
}
$$

meaning both directions are possible.

A chain is irreducible when every pair of states communicates.


In [ ]:
P_classes = np.array([
    [0.5,0.5,0.0,0.0],
    [0.4,0.6,0.0,0.0],
    [0.0,0.0,0.3,0.7],
    [0.0,0.0,0.2,0.8],
])

display(Markdown(
    f"Communication classes: **{communication_classes(P_classes)}**"
))
display(Markdown(
    f"Irreducible: **{is_irreducible(P_classes)}**"
))


## 7. Recurrence and transience

Let

$$
f_i
=
P_i(T_i^+<\infty).
$$

State $i$ is:

- recurrent if $f_i=1$;
- transient if $f_i<1$;
- positive recurrent if it is recurrent and

$$
\mathbb E_i[T_i^+]<\infty;
$$

- null recurrent if the return probability is one but the mean return time is infinite.


### Recurrence criterion

For every state $i$,

$$
\boxed{
i\text{ recurrent}
\iff
\sum_{n=0}^{\infty}
(P^n)_{ii}
=
\infty.
}
$$

If $i$ is transient, then the expected total number of visits to $i$, including time zero, is

$$
\boxed{
\frac1{1-f_i}.
}
$$


### Recurrence is a class property

If

$$
i\leftrightarrow j,
$$

then $i$ is recurrent if and only if $j$ is recurrent.

Therefore an irreducible chain is either recurrent in every state or transient in every state.


### Finite irreducible chains

Every state of a finite irreducible chain is positive recurrent.

Moreover,

$$
\boxed{
\mathbb E_j[T_i]<\infty
}
$$

for every pair of states $i,j$.


## 8. Stationary distributions

A probability row vector $\pi$ is stationary if

$$
\boxed{
\pi P=\pi.
}
$$

If

$$
X_0\sim\pi,
$$

then

$$
X_n\sim\pi
$$

for every $n$.


Every finite irreducible chain has a unique stationary distribution, and

$$
\boxed{
\pi_j>0
}
$$

for every state $j$.


In [ ]:
P_stationary = np.array([
    [0.8,0.2],
    [0.3,0.7],
])

pi_stationary = stationary_distribution(P_stationary)

display(Markdown(
    f"Stationary distribution: **{np.round(pi_stationary,6)}**"
))
display(Markdown(
    f"Stationarity error: **{np.max(np.abs(pi_stationary@P_stationary-pi_stationary)):.3e}**"
))


### Two-state formula

For

$$
P=
\begin{pmatrix}
1-a&a\\
b&1-b
\end{pmatrix},
\qquad
a,b>0,
$$

the stationary distribution is

$$
\boxed{
\pi
=
\left(
\frac{b}{a+b},
\frac{a}{a+b}
\right).
}
$$


### Kac's mean return-time formula

For a finite irreducible chain,

$$
\boxed{
\mathbb E_i[T_i^+]
=
\frac1{\pi_i}.
}
$$

Large stationary mass means short mean return time.


In [ ]:
a,b = 0.2,0.3

pi1 = b/(a+b)
pi2 = a/(a+b)

display(Math(r"\pi_1=" + f"{pi1:.6f}"))
display(Math(r"\mathbb E_1[T_1^+]=" + f"{1/pi1:.6f}"))
display(Math(r"\pi_2=" + f"{pi2:.6f}"))
display(Math(r"\mathbb E_2[T_2^+]=" + f"{1/pi2:.6f}"))


## 9. Reversibility and detailed balance

A chain is reversible with respect to $\pi$ when

$$
\boxed{
\pi_ip_{ij}
=
\pi_jp_{ji}
}
$$

for all states $i,j$.

These are the detailed-balance equations.


Detailed balance implies stationarity:

$$
\boxed{
\text{detailed balance}
\Longrightarrow
\pi P=\pi.
}
$$

The converse is false in general.

Stationarity balances total flow into and out of each state; detailed balance requires pairwise flow balance on every edge.


### Random walk on a finite undirected graph

If the walk chooses uniformly among neighboring vertices, then

$$
\boxed{
\pi_i
=
\frac{\deg(i)}{2|E|}.
}
$$

Indeed, for adjacent vertices,

$$
\pi_ip_{ij}
=
\frac1{2|E|}
=
\pi_jp_{ji}.
$$


In [ ]:
# Path graph 1-2-3-4.
P_path = np.array([
    [0,1,0,0],
    [0.5,0,0.5,0],
    [0,0.5,0,0.5],
    [0,0,1,0],
], dtype=float)

pi_path = np.array([1,2,2,1],dtype=float)
pi_path /= pi_path.sum()

display(Markdown(
    f"Stationary degree distribution: **{pi_path}**"
))
display(Markdown(
    f"Detailed-balance error: **{detailed_balance_error(P_path,pi_path):.3e}**"
))


## 10. Periodicity

For a state $i$, let

$$
\mathcal R_i
=
\{n\ge1:(P^n)_{ii}>0\}.
$$

If $\mathcal R_i\ne\varnothing$, define

$$
\boxed{
d(i)
=
\gcd\mathcal R_i.
}
$$

The state is aperiodic if

$$
d(i)=1.
$$


### Immediate tests

If

$$
p_{ii}>0,
$$

then

$$
d(i)=1.
$$

More generally, if two possible return times $r,s$ satisfy

$$
\gcd(r,s)=1,
$$

then the state is aperiodic.


### Communication preserves period

Communicating states have the same period.

Thus an irreducible Markov chain has one common period

$$
\boxed{
d.
}
$$


In [ ]:
P_alt = np.array([
    [0,1],
    [1,0],
], dtype=float)

display(Markdown(
    f"Perfect alternation period: **{period_irreducible(P_alt)}**"
))


## 11. Graph formula for the period

For a finite irreducible chain, choose a reference state $r$ and one directed path from $r$ to every state $j$, with length $\ell(j)$.

Then

$$
\boxed{
d
=
\gcd
\{
|\ell(i)+1-\ell(j)|:
p_{ij}>0,\
\ell(i)+1-\ell(j)\ne0
\}.
}
$$

This converts the infinite return-time definition into a finite graph calculation.


### Aperiodic without a self-loop

Consider positive-probability edges

$$
1\to2,
\qquad
2\to3,
\qquad
3\to1,
\qquad
3\to2.
$$

The graph contains return cycles of lengths $3$ and $2$.

Therefore

$$
\gcd(2,3)=1,
$$

so the chain is aperiodic even though it has no self-loop.


In [ ]:
P_graph = np.array([
    [0,1,0],
    [0,0,1],
    [0.5,0.5,0],
], dtype=float)

display(Markdown(
    f"Irreducible: **{is_irreducible(P_graph)}**"
))
display(Markdown(
    f"Period: **{period_irreducible(P_graph)}**"
))


## 12. Cyclic classes

For an irreducible chain of period $d$, states split into

$$
\boxed{
S
=
C_0\cup\cdots\cup C_{d-1},
}
$$

and every positive one-step transition moves cyclically:

$$
\boxed{
C_0
\to
C_1
\to
\cdots
\to
C_{d-1}
\to
C_0.
}
$$


If the chain is finite and irreducible with stationary distribution $\pi$, then every cyclic class has the same stationary mass:

$$
\boxed{
\pi(C_a)
=
\frac1d.
}
$$

This factor $1/d$ explains the factor $d$ appearing in periodic subsequence limits.


In [ ]:
P_cycle3 = np.array([
    [0,1,0],
    [0,0,1],
    [1,0,0],
], dtype=float)

classes3 = cyclic_classes(P_cycle3)
pi3 = stationary_distribution(P_cycle3)

display(Markdown(f"Cyclic classes: **{classes3}**"))
display(Markdown(f"Stationary distribution: **{pi3}**"))
display(Markdown(
    f"Stationary masses by cyclic class: **{[pi3[c].sum() for c in classes3]}**"
))


## 13. Aperiodic convergence theorem

For a finite irreducible aperiodic chain,

$$
\boxed{
(P^n)_{ij}
\longrightarrow
\pi_j
}
$$

for all $i,j$.

Equivalently,

$$
\boxed{
P^n
\longrightarrow
\mathbf1\pi.
}
$$


### Eventual positivity

Finite irreducibility plus aperiodicity implies that there is an $N$ such that

$$
\boxed{
(P^n)_{ij}>0
}
$$

for every pair $i,j$ and every $n\ge N$.

The chapter uses this to produce an $L^1$ contraction argument for convergence to stationarity.


In [ ]:
P_ap = np.array([
    [0.7,0.3],
    [0.2,0.8],
], dtype=float)

pi_ap = stationary_distribution(P_ap)
target_ap = np.ones((2,1)) @ pi_ap.reshape(1,-1)

ap_n = widgets.IntSlider(
    value=10,
    min=1,
    max=40,
    description="n",
)
ap_output = widgets.Output()


def update_aperiodic(*_):
    with ap_output:
        clear_output(wait=True)

        n = ap_n.value
        Pn = matrix_power(P_ap,n)

        display(Markdown(
            f"$P^{n}$ =\n\n```\n{np.round(Pn,6)}\n```"
        ))
        display(Markdown(
            f"Max error from $1\\pi$: **{np.max(np.abs(Pn-target_ap)):.6g}**"
        ))


ap_n.observe(update_aperiodic, names="value")
display(widgets.VBox([ap_n,ap_output]))
update_aperiodic()


## 14. Periodic chains: all residue-class limits

Let a finite irreducible chain have period $d$.

If $i\in C_0$ and $j\in C_a$, then

$$
\boxed{
\lim_{n\to\infty}
(P^{nd+a})_{ij}
=
d\pi_j.
}
$$

For every other residue $b\ne a$,

$$
\boxed{
(P^{nd+b})_{ij}=0.
}
$$


Define

$$
\boxed{
L_a
=
\lim_{n\to\infty}
P^{nd+a}.
}
$$

If state $i$ lies in $C_r$, then

$$
\boxed{
(L_a)_{ij}
=
\begin{cases}
d\pi_j,
&j\in C_{r+a\pmod d},\\
0,
&\text{otherwise}.
\end{cases}
}
$$

Also,

$$
\boxed{
L_a=P^aL_0.
}
$$


In [ ]:
P_period2 = np.array([
    [0.0,0.5,0.0,0.5],
    [0.5,0.0,0.5,0.0],
    [0.0,0.5,0.0,0.5],
    [0.5,0.0,0.5,0.0],
])

pi_period2 = stationary_distribution(P_period2)
d_period2 = period_irreducible(P_period2)

L0 = matrix_power(P_period2,200)
L1 = matrix_power(P_period2,201)

display(Markdown(f"Period: **{d_period2}**"))
display(Markdown(f"Stationary distribution: **{pi_period2}**"))
display(Markdown(f"$L_0$:\n\n```\n{np.round(L0,4)}\n```"))
display(Markdown(f"$L_1$:\n\n```\n{np.round(L1,4)}\n```"))


### All subsequential limits

For a finite irreducible period-$d$ chain, the complete set of accumulation matrices of $(P^n)$ is

$$
\boxed{
\{L_0,\ldots,L_{d-1}\},
}
$$

with repetitions removed if two limits happen to coincide.

Every convergent subsequence must eventually contain a further subsequence in one fixed residue class modulo $d$.


## 15. Countably infinite state spaces: multiplication order matters

Suppose a finite or countably infinite chain satisfies

$$
Q_\infty
=
\lim_{n\to\infty}
P^{nd}
$$

entrywise.

Then for fixed $k$,

$$
\boxed{
P^{nd+k}
\longrightarrow
P^kQ_\infty.
}
$$

The proof uses dominated convergence with the fixed row probabilities of $P^k$.


In an infinite state space, entrywise convergence alone does **not** automatically justify

$$
P^{nd+k}
\longrightarrow
Q_\infty P^k.
$$

The coefficients in the infinite sum can depend on $n$, and mass may escape to infinity.

An entrywise limit of stochastic matrices may even be only substochastic.


## 16. Cesàro averaging removes periodic oscillation

If the residue-class limits exist, then

$$
\boxed{
\lim_{N\to\infty}
\frac1N
\sum_{n=0}^{N-1}
P^n
=
\frac1d
\sum_{a=0}^{d-1}
L_a.
}
$$

For a finite irreducible chain,

$$
\boxed{
\frac1N
\sum_{n=0}^{N-1}
P^n
\longrightarrow
\mathbf1\pi.
}
$$


For finite irreducible chains,

$$
\boxed{
\mathbf1\pi
=
\frac1d
\sum_{a=0}^{d-1}
L_a.
}
$$

For each fixed pair $(i,j)$, exactly one of the $d$ residue-class limits equals

$$
d\pi_j,
$$

while the other $d-1$ limits are zero. Their average is therefore $\pi_j$.


In [ ]:
cesaro_N = widgets.IntSlider(
    value=500,
    min=10,
    max=5000,
    step=10,
    description="N",
)
cesaro_output = widgets.Output()


def update_cesaro(*_):
    with cesaro_output:
        clear_output(wait=True)

        N = cesaro_N.value

        C = cesaro_average(
            P_period2,
            N,
        )

        target = np.ones((4,1)) @ pi_period2.reshape(1,-1)

        display(Markdown(
            f"Max Cesàro error from $1\\pi$: **{np.max(np.abs(C-target)):.6g}**"
        ))
        display(Markdown(
            f"Max average-phase error from $1\\pi$: **{np.max(np.abs((L0+L1)/2-target)):.6g}**"
        ))


cesaro_N.observe(update_cesaro, names="value")
display(widgets.VBox([cesaro_N,cesaro_output]))
update_cesaro()


## 17. Sample-path ergodic theorem

For every finite irreducible chain and every function $g:S\to\mathbb R$,

$$
\boxed{
\frac1N
\sum_{n=0}^{N-1}
g(X_n)
\xrightarrow{\mathrm{a.s.}}
\sum_jg(j)\pi_j.
}
$$

No aperiodicity assumption is required.


In particular, empirical occupation frequencies satisfy

$$
\boxed{
\frac1N
\sum_{n=0}^{N-1}
\mathbf1_{\{X_n=j\}}
\xrightarrow{\mathrm{a.s.}}
\pi_j.
}
$$

Thus a periodic chain can have oscillating one-time distributions while a single long trajectory still has stable long-run frequencies.


In [ ]:
erg_N = widgets.IntSlider(
    value=50000,
    min=1000,
    max=300000,
    step=1000,
    description="N",
)
erg_output = widgets.Output()


def update_ergodic_sim(*_):
    with erg_output:
        clear_output(wait=True)

        N = erg_N.value

        states = simulate_chain(
            P_period2,
            x0=0,
            N=N,
            seed=2026,
        )

        freq = np.bincount(
            states,
            minlength=4,
        )/N

        display(Markdown(
            f"Empirical occupation frequencies: **{np.round(freq,4)}**"
        ))
        display(Markdown(
            f"Stationary distribution: **{np.round(pi_period2,4)}**"
        ))


erg_N.observe(update_ergodic_sim, names="value")
display(widgets.VBox([erg_N,erg_output]))
update_ergodic_sim()


### Deterministic three-cycle

For

$$
1\to2\to3\to1\to\cdots,
$$

the law of $X_n$ never converges from a fixed starting state.

Nevertheless each state appears once every three steps, so its empirical occupation frequency converges to

$$
\boxed{
1/3.
}
$$


## 18. Two periodic examples

### Perfect period-two alternation

For

$$
P=
\begin{pmatrix}
0&1\\
1&0
\end{pmatrix},
$$

$$
P^{2n}=I,
\qquad
P^{2n+1}=P.
$$

The stationary distribution is

$$
\pi=(1/2,1/2),
$$

but $P^n$ does not converge.


In [ ]:
P_flip = np.array([
    [0,1],
    [1,0],
], dtype=float)

pi_flip = stationary_distribution(P_flip)

display(Markdown(f"$P^{{20}}$:\n\n```\n{matrix_power(P_flip,20)}\n```"))
display(Markdown(f"$P^{{21}}$:\n\n```\n{matrix_power(P_flip,21)}\n```"))
display(Markdown(f"Stationary distribution: **{pi_flip}**"))


### Three-cycle

For

$$
P=
\begin{pmatrix}
0&1&0\\
0&0&1\\
1&0&0
\end{pmatrix},
$$

$$
P^{3n}=I,
$$

$$
P^{3n+1}=P,
$$

$$
P^{3n+2}=P^2.
$$

Thus the complete set of subsequential limits is

$$
\boxed{
\{I,P,P^2\}.
}
$$

Their average is

$$
\boxed{
\frac{I+P+P^2}{3}
=
\mathbf1\pi.
}
$$


In [ ]:
C3 = (
    np.eye(3)
    + P_cycle3
    + matrix_power(P_cycle3,2)
)/3

target3 = np.ones((3,1)) @ pi3.reshape(1,-1)

display(Markdown(
    f"Max discrepancy: **{np.max(np.abs(C3-target3)):.3e}**"
))


## 19. Historical problem: gambler's ruin

A gambler has fortune

$$
0,1,\ldots,N.
$$

At every round the fortune increases by one with probability $p$ and decreases by one with probability

$$
q=1-p.
$$

The process stops at $0$ or $N$.

Let

$$
h_i
=
P_i(T_N<T_0).
$$

First-step analysis gives

$$
\boxed{
h_i
=
ph_{i+1}
+
qh_{i-1}.
}
$$


### Exact gambler's-ruin probability

For $1\le i\le N-1$,

$$
\boxed{
h_i
=
\begin{cases}
0,
&p=0,\\
\dfrac{
1-(q/p)^i
}{
1-(q/p)^N
},
&0<p<1,\ p\ne1/2,\\
\dfrac{i}{N},
&p=1/2,\\
1,
&p=1.
\end{cases}
}
$$


In [ ]:
ruin_N = widgets.IntSlider(
    value=10,
    min=2,
    max=100,
    description="N",
)
ruin_i = widgets.IntSlider(
    value=4,
    min=1,
    max=9,
    description="i",
)
ruin_p = widgets.FloatSlider(
    value=0.6,
    min=0,
    max=1,
    step=0.05,
    description="p",
)
ruin_output = widgets.Output()


def update_ruin(*_):
    with ruin_output:
        clear_output(wait=True)

        N = ruin_N.value

        if ruin_i.max != N-1:
            ruin_i.max = N-1

        i = min(ruin_i.value,N-1)
        p = ruin_p.value

        h = gambler_ruin_probability(
            i,
            N,
            p,
        )

        m = gambler_ruin_duration(
            i,
            N,
            p,
        )

        display(Math(r"h_i=" + f"{h:.8f}"))
        display(Math(r"\mathbb E_i[\tau]=" + f"{m:.8f}"))


for control in (ruin_N,ruin_i,ruin_p):
    control.observe(update_ruin, names="value")

display(widgets.VBox([
    widgets.HBox([ruin_N,ruin_i,ruin_p]),
    ruin_output,
]))
update_ruin()


For the biased example

$$
N=10,
\qquad
i=4,
\qquad
p=0.6,
$$

we obtain

$$
\boxed{
h_4
=
\frac{
1-(2/3)^4
}{
1-(2/3)^{10}
}
\approx0.8166.
}
$$


## 20. Expected duration of gambler's ruin

Let

$$
\tau=T_{\{0,N\}},
$$

and

$$
m_i=\mathbb E_i[\tau].
$$

The first-step equation is

$$
\boxed{
m_i
=
1
+
pm_{i+1}
+
qm_{i-1},
}
$$

with

$$
m_0=m_N=0.
$$


For $0<p<1$,

$$
\boxed{
m_i
=
\begin{cases}
i(N-i),
&p=q=1/2,\\
\dfrac{i}{q-p}
-
\dfrac{N}{q-p}
\dfrac{
1-(q/p)^i
}{
1-(q/p)^N
},
&p\ne q.
\end{cases}
}
$$

For $p=0$,

$$
m_i=i,
$$

and for $p=1$,

$$
m_i=N-i.
$$


### A fair game can last a long time

For boundaries $0$ and $100$, starting from $30$,

$$
\boxed{
P_{30}(T_{100}<T_0)
=
0.3,
}
$$

but

$$
\boxed{
\mathbb E_{30}[\tau]
=
30(70)
=
2100.
}
$$


In [ ]:
display(Math(
    r"h_{30}="
    + f"{gambler_ruin_probability(30,100,0.5):.6f}"
))
display(Math(
    r"m_{30}="
    + f"{gambler_ruin_duration(30,100,0.5):.0f}"
))


In [ ]:
gr_sim_N = widgets.IntSlider(
    value=20000,
    min=1000,
    max=100000,
    step=1000,
    description="reps",
)
gr_sim_output = widgets.Output()


def update_gambler_sim(*_):
    with gr_sim_output:
        clear_output(wait=True)

        reps = gr_sim_N.value
        rng = np.random.default_rng(2026)

        successes = 0
        durations = []

        for _ in range(reps):
            x = 30
            t = 0

            while x not in (0,100):
                x += 1 if rng.random() < 0.5 else -1
                t += 1

            successes += int(x == 100)
            durations.append(t)

        display(Math(
            r"\widehat P(T_{100}<T_0)="
            + f"{successes/reps:.6f}"
        ))
        display(Math(
            r"\widehat{\mathbb E}[\tau]="
            + f"{np.mean(durations):.3f}"
        ))
        display(Math(r"\text{exact probability}=0.3"))
        display(Math(r"\text{exact mean duration}=2100"))


gr_sim_N.observe(update_gambler_sim, names="value")
display(widgets.VBox([gr_sim_N,gr_sim_output]))
update_gambler_sim()


## 21. Random walks as Markov chains

Let

$$
P(\xi_n=1)=p,
\qquad
P(\xi_n=-1)=q=1-p,
$$

with independent increments, and define

$$
S_n
=
\xi_1+\cdots+\xi_n.
$$

Then $(S_n)$ is a Markov chain on $\mathbb Z$ with

$$
\boxed{
p_{ij}
=
\begin{cases}
p,
&j=i+1,\\
q,
&j=i-1,\\
0,
&\text{otherwise}.
\end{cases}
}
$$


### Recurrence of the one-dimensional simple random walk

For

$$
0<p<1,
$$

the walk is recurrent if

$$
\boxed{
p=q=1/2,
}
$$

and transient if

$$
\boxed{
p\ne q.
}
$$

For $p>q$, the return probability after leaving zero is

$$
\boxed{
P_0(T_0^+<\infty)
=
2q.
}
$$

For $p<q$, the symmetric expression is $2p$.


For example, if

$$
p=0.6,
\qquad
q=0.4,
$$

then

$$
\boxed{
P_0(T_0^+<\infty)
=
0.8.
}
$$

There is a $20\%$ chance that the walk never returns to its starting state after time zero.


In [ ]:
rw_p = widgets.FloatSlider(
    value=0.6,
    min=0.05,
    max=0.95,
    step=0.05,
    description="p",
)
rw_output = widgets.Output()


def update_random_walk_type(*_):
    with rw_output:
        clear_output(wait=True)

        p = rw_p.value
        q = 1-p

        if abs(p-0.5) < 1e-12:
            display(Markdown("**Recurrent.**"))
            display(Math(r"P_0(T_0^+<\infty)=1"))
        else:
            return_prob = 2*min(p,q)

            display(Markdown("**Transient.**"))
            display(Math(
                r"P_0(T_0^+<\infty)="
                + f"{return_prob:.6f}"
            ))


rw_p.observe(update_random_walk_type, names="value")
display(widgets.VBox([rw_p,rw_output]))
update_random_walk_type()


## 22. Python laboratory: periodicity and long-run phases

The following integrated experiment compares:

- the stationary distribution;
- the graph-computed period;
- residue-class limits;
- the Cesàro average;
- detailed balance;
- sample-path occupation frequencies.

The example is the chapter's period-two four-state walker.


In [ ]:
lab_n = widgets.IntSlider(
    value=1000,
    min=50,
    max=5000,
    step=50,
    description="Cesaro N",
)
lab_path_N = widgets.IntSlider(
    value=50000,
    min=1000,
    max=300000,
    step=1000,
    description="path N",
)
lab_output = widgets.Output()


def update_markov_lab(*_):
    with lab_output:
        clear_output(wait=True)

        N = lab_n.value
        path_N = lab_path_N.value

        P = P_period2
        pi = stationary_distribution(P)
        d = period_irreducible(P)
        classes = cyclic_classes(P)

        base = 200*d
        limits = [
            matrix_power(P,base+a)
            for a in range(d)
        ]

        average_limits = sum(limits)/d
        C = cesaro_average(P,N)
        target = np.ones((P.shape[0],1)) @ pi.reshape(1,-1)

        states = simulate_chain(
            P,
            x0=0,
            N=path_N,
            seed=2026,
        )

        freq = np.bincount(
            states,
            minlength=P.shape[0],
        )/path_N

        display(Markdown(f"Stationary distribution: **{np.round(pi,6)}**"))
        display(Markdown(f"Period: **{d}**"))
        display(Markdown(f"Cyclic classes: **{classes}**"))
        display(Markdown(
            f"Max phase-average error: **{np.max(np.abs(average_limits-target)):.3e}**"
        ))
        display(Markdown(
            f"Max Cesàro error: **{np.max(np.abs(C-target)):.3e}**"
        ))
        display(Markdown(
            f"Detailed-balance error: **{detailed_balance_error(P,pi):.3e}**"
        ))
        display(Markdown(
            f"Occupation frequencies: **{np.round(freq,4)}**"
        ))


for control in (lab_n,lab_path_N):
    control.observe(update_markov_lab, names="value")

display(widgets.VBox([
    widgets.HBox([lab_n,lab_path_N]),
    lab_output,
]))
update_markov_lab()


### Generating a random positive stochastic matrix

A matrix with all strictly positive entries is automatically irreducible and aperiodic.

The next cell creates such a matrix, computes its stationary distribution, and compares a large matrix power with $\mathbf1\pi$.


In [ ]:
rand_m = widgets.IntSlider(
    value=4,
    min=2,
    max=8,
    description="states",
)
rand_seed = widgets.IntSlider(
    value=123,
    min=0,
    max=5000,
    description="seed",
)
rand_output = widgets.Output()


def update_random_matrix(*_):
    with rand_output:
        clear_output(wait=True)

        m = rand_m.value
        seed = rand_seed.value

        rng = np.random.default_rng(seed)
        A = rng.random((m,m))
        P = A/A.sum(axis=1,keepdims=True)

        pi = stationary_distribution(P)
        d = period_irreducible(P)

        target = np.ones((m,1)) @ pi.reshape(1,-1)
        Pn = matrix_power(P,100)

        display(Markdown(f"Period: **{d}**"))
        display(Markdown(
            f"Maximum $P^{{100}}-1\\pi$ error: **{np.max(np.abs(Pn-target)):.3e}**"
        ))


for control in (rand_m,rand_seed):
    control.observe(update_random_matrix, names="value")

display(widgets.VBox([
    widgets.HBox([rand_m,rand_seed]),
    rand_output,
]))
update_random_matrix()


## 23. Solved-style computational checks


### Two-step transition probability

For

$$
P=
\begin{pmatrix}
0.7&0.3\\
0.2&0.8
\end{pmatrix},
$$

$$
P^2
=
\begin{pmatrix}
0.55&0.45\\
0.30&0.70
\end{pmatrix}.
$$

Therefore

$$
\boxed{
P_1(X_2=2)=0.45.
}
$$


In [ ]:
P_ex = np.array([
    [0.7,0.3],
    [0.2,0.8],
])

P2_ex = matrix_power(P_ex,2)

display(Markdown(f"$P^2$:\n\n```\n{P2_ex}\n```"))
display(Math(r"P_1(X_2=2)=" + f"{P2_ex[0,1]:.2f}"))


### Detailed balance on a path graph

For the path

$$
1-2-3-4,
$$

degrees are

$$
1,2,2,1.
$$

Therefore

$$
\boxed{
\pi
=
\left(
\frac16,
\frac13,
\frac13,
\frac16
\right).
}
$$


### Period-four recovery exercise

If a finite irreducible chain has period $4$ and

$$
\lim_{n\to\infty}
(P^{4n+2})_{ij}
=
0.28
$$

while all other residue-class limits for the fixed pair $(i,j)$ are zero, then the nonzero limit equals

$$
4\pi_j.
$$

Therefore

$$
\boxed{
\pi_j=0.07.
}
$$


### Period-two scalar limit

If

$$
i\in C_0,
\qquad
j\in C_1,
\qquad
\pi_j=0.12,
$$

then

$$
\boxed{
\lim_{n\to\infty}
(P^{2n})_{ij}=0,
}
$$

and

$$
\boxed{
\lim_{n\to\infty}
(P^{2n+1})_{ij}
=
2(0.12)
=
0.24.
}
$$


## 24. Guided exercise generator


In [ ]:
exercise_rng = random.Random(20260815)

exercise_kind = widgets.Dropdown(
    options=[
        ("Random","random"),
        ("Transition matrix","transition"),
        ("Markov operator","operator"),
        ("Hitting time","hitting"),
        ("Stationarity","stationary"),
        ("Kac","kac"),
        ("Period","period"),
        ("Periodic limit","limit"),
        ("Gambler's ruin","ruin"),
        ("Random walk","walk"),
    ],
    value="random",
    description="Type",
)

new_button = widgets.Button(description="New exercise")
hint_button = widgets.Button(description="Hint")
reveal_button = widgets.Button(description="Reveal")
check_button = widgets.Button(description="Check")
answer_box = widgets.Text(description="Answer")
prompt_output = widgets.Output()
feedback_output = widgets.Output()
state = {}


def make_exercise(_=None):
    kind = exercise_kind.value

    if kind == "random":
        kind = exercise_rng.choice([
            "transition",
            "operator",
            "hitting",
            "stationary",
            "kac",
            "period",
            "limit",
            "ruin",
            "walk",
        ])

    if kind == "transition":
        target = "1"
        prompt = "What must every row of a transition matrix sum to?"
        hint = "Each row is a conditional probability distribution."
        solution = r"\text{Each row sums to }1."

    elif kind == "operator":
        target = "2"
        prompt = "For P=[[0.8,0.2],[0.3,0.7]] and g=(0,10), find (Pg)(1)."
        hint = "Take the first-row weighted average."
        solution = r"(Pg)(1)=2."

    elif kind == "hitting":
        target = "2"
        prompt = "At state 1, the chain stays with probability 0.5 and otherwise exits immediately. Find the expected time to exit."
        hint = "m=1+0.5m."
        solution = r"m=2."

    elif kind == "stationary":
        target = "0.6"
        prompt = "For P=[[0.8,0.2],[0.3,0.7]], find stationary probability pi_1."
        hint = "Solve pi P=pi and sum pi_i=1."
        solution = r"\pi_1=0.6."

    elif kind == "kac":
        target = "50"
        prompt = "A state has stationary probability 0.02 in a finite irreducible chain. Find its mean positive return time."
        hint = "Use Kac's formula."
        solution = r"\mathbb E_i[T_i^+]=50."

    elif kind == "period":
        target = "2"
        prompt = "For deterministic alternation 1<->2, find the period."
        hint = "Returns occur at 2,4,6,..."
        solution = r"d=2."

    elif kind == "limit":
        target = "0.24"
        prompt = "Period d=2, i in C0, j in C1, pi_j=0.12. Find lim (P^(2n+1))_ij."
        hint = "The allowed residue-class limit is d pi_j."
        solution = r"2\pi_j=0.24."

    elif kind == "ruin":
        target = "0.3"
        prompt = "Fair gambler's ruin with boundaries 0 and 100, starting at 30. Find the probability of reaching 100 first."
        hint = "For p=1/2, h_i=i/N."
        solution = r"h_{30}=0.3."

    else:
        target = "transient"
        prompt = "A one-dimensional simple random walk has p=0.6. Is it recurrent or transient?"
        hint = "Only the balanced p=1/2 walk is recurrent."
        solution = r"\text{Transient.}"

    state.clear()
    state.update(
        target=target,
        hint=hint,
        solution=solution,
    )

    answer_box.value = ""

    with prompt_output:
        clear_output(wait=True)
        display(Markdown("### Exercise\n" + prompt))

    with feedback_output:
        clear_output(wait=True)


def show_hint(_):
    with feedback_output:
        clear_output(wait=True)
        display(Markdown("**Hint:** " + state["hint"]))


def reveal(_):
    with feedback_output:
        clear_output(wait=True)
        display(Math(state["solution"]))


def check(_):
    with feedback_output:
        clear_output(wait=True)

        guess = answer_box.value.strip().lower().replace(" ","")
        target = state["target"].strip().lower().replace(" ","")

        correct = guess == target

        if not correct:
            try:
                correct = abs(float(guess)-float(target)) < 5e-4
            except Exception:
                pass

        display(Markdown(
            "**Correct.**"
            if correct
            else "**Not yet. Identify the Markov-chain structure and theorem hypotheses first.**"
        ))


new_button.on_click(make_exercise)
hint_button.on_click(show_hint)
reveal_button.on_click(reveal)
check_button.on_click(check)

display(widgets.VBox([
    widgets.HBox([exercise_kind,new_button]),
    prompt_output,
    widgets.HBox([answer_box,check_button]),
    widgets.HBox([hint_button,reveal_button]),
    feedback_output,
]))

make_exercise()


## 25. AI Audit: Markov-chain claims

Use this checklist on any AI-generated Markov-chain argument.

1. Is the transition matrix row-stochastic?
2. Is the Markov property stated conditionally on the current state rather than as independence?
3. Are zero-probability histories handled carefully?
4. Is the natural filtration $\mathcal F_n=\sigma(X_0,\ldots,X_n)$ used consistently?
5. Is $(Pg)(i)$ computed as a row-weighted expectation?
6. Is $(P^n)_{ij}$ interpreted under the chain started from $i$?
7. Does a path probability include the initial-state probability?
8. Is Chapman--Kolmogorov used as matrix multiplication through an intermediate state?
9. Is a hitting time distinguished from a positive return time?
10. Is the stopping-time event $\{\tau\le n\}$ measurable using information available by time $n$?
11. Are first-step equations supplied with the correct boundary values?
12. Is the strong Markov property used only at a stopping time?
13. Is communication distinguished from one-way accessibility?
14. Is recurrence treated as a class property?
15. Is a finite irreducible chain recognized as positive recurrent?
16. Is a stationary distribution solved from $\pi P=\pi$ and $\sum_i\pi_i=1$?
17. Is stationarity being confused with convergence of $P^n$?
18. Is Kac's formula written as $E_i[T_i^+]=1/\pi_i$?
19. Is detailed balance treated as sufficient but not necessary for stationarity?
20. Is the period computed as a gcd of possible positive return times?
21. Is a self-loop correctly recognized as proving period one?
22. Are communicating states assigned the same period?
23. Are cyclic classes advanced one step at a time?
24. Is each cyclic class assigned stationary mass $1/d$ in the finite irreducible case?
25. Is aperiodicity required for $P^n\to\mathbf1\pi$?
26. For a period-$d$ chain, is the allowed residue-class limit $d\pi_j$ rather than $\pi_j$?
27. Are all residue-class limits distinguished from the full sequence limit?
28. In a countably infinite chain, is the justified formula $P^kQ_\infty$ used in the correct multiplication order?
29. Is the Cesàro limit distinguished from the ordinary matrix-power limit?
30. Is periodicity correctly recognized as compatible with sample-path ergodic convergence?
31. Is the gambler's-ruin boundary-value recurrence derived before inserting a closed-form solution?
32. In the fair ruin problem, is $h_i=i/N$?
33. Is the fair expected duration $i(N-i)$?
34. Is the one-dimensional simple random walk recurrent only when $p=1/2$?
35. Is simulation being used as illustration rather than proof of universal Markov-chain theorems?

### Claims to audit

- “The Markov property means $X_n$ and $X_{n+1}$ are independent.”
- “Every finite irreducible chain has $P^n\to\mathbf1\pi$.”
- “If a stationary distribution exists, the chain must converge to it from every initial state.”
- “Detailed balance is necessary for stationarity.”
- “A periodic chain cannot satisfy a sample-path ergodic theorem.”
- “If $\pi_i=0.02$, then $E_i[T_i^+]=0.02$.”

All six claims are false.


### Repairs

The Markov property is conditional: once $X_n$ is known, the earlier path adds no information about the next transition.

Finite irreducibility gives a unique stationary distribution, but ordinary convergence of $P^n$ also requires aperiodicity.

Detailed balance is one convenient sufficient condition for stationarity, not a necessary one.

Periodicity can block convergence of one-time distributions while still allowing both Cesàro and sample-path ergodic averages to converge.

Kac's formula gives

$$
E_i[T_i^+]
=
1/\pi_i,
$$

so $\pi_i=0.02$ corresponds to mean return time $50$.


### Suggested AI-guided activities

- “Give me a small transition matrix and make me verify stochasticity, compute $P^2$, find communication classes and solve for $\pi$ one step at a time.”
- “Teach me period through cyclic classes. Make me predict which entries of $P^{nd+a}$ must be zero before calculating any limit.”
- “Give me a period-two chain and make me compute $L_0$, $L_1$, their average, and the Cesàro limit.”
- “Give me a target set and make me derive both hitting-probability and expected-hitting-time first-step equations.”
- “Give me a graph random walk and make me derive the stationary law from detailed balance.”
- “Give me a stationary probability and make me interpret it through Kac's formula.”
- “Give me a biased gambler's-ruin problem and require both the success probability and expected absorption time.”


## 26. Self-check quiz


In [ ]:
quiz_data = [
    (
        "1. The Markov property says consecutive states are independent:",
        ["Choose...","true","false"],
        "false",
        r"\text{The dependence is conditional on the present state.}",
    ),
    (
        "2. For a finite chain started from i, (P^n)_ij equals:",
        ["Choose...","P_i(X_n=j)","P(X_0=i,X_n=j) always"],
        "P_i(X_n=j)",
        r"(P^n)_{ij}=P_i(X_n=j).",
    ),
    (
        "3. A hitting time is a stopping time:",
        ["Choose...","true","false"],
        "true",
        r"\{T_A\le n\}=\bigcup_{r=0}^n\{X_r\in A\}.",
    ),
    (
        "4. Every finite irreducible chain is positive recurrent:",
        ["Choose...","true","false"],
        "true",
        r"\text{Finite irreducibility implies finite mean hitting and return times.}",
    ),
    (
        "5. Detailed balance is necessary for stationarity:",
        ["Choose...","true","false"],
        "false",
        r"\text{It is sufficient, not necessary.}",
    ),
    (
        "6. Kac's formula gives mean return time:",
        ["Choose...","1/pi_i","pi_i","pi_i^2"],
        "1/pi_i",
        r"\mathbb E_i[T_i^+]=1/\pi_i.",
    ),
    (
        "7. A self-loop in an irreducible chain implies:",
        ["Choose...","aperiodicity","period two","transience"],
        "aperiodicity",
        r"1\in\mathcal R_i\Longrightarrow d=1.",
    ),
    (
        "8. In a finite irreducible period-d chain, each cyclic class has stationary mass:",
        ["Choose...","1/d","d","pi_i"],
        "1/d",
        r"\pi(C_a)=1/d.",
    ),
    (
        "9. A finite irreducible periodic chain must have P^n convergent:",
        ["Choose...","true","false"],
        "false",
        r"\text{Periodic residue classes can oscillate.}",
    ),
    (
        "10. The Cesaro average of a finite irreducible chain converges to:",
        ["Choose...","1*pi","0","P"],
        "1*pi",
        r"\frac1N\sum_{n=0}^{N-1}P^n\to\mathbf1\pi.",
    ),
    (
        "11. The sample-path ergodic theorem requires aperiodicity:",
        ["Choose...","true","false"],
        "false",
        r"\text{Finite irreducibility is enough here.}",
    ),
    (
        "12. The fair gambler's-ruin success probability from i is:",
        ["Choose...","i/N","1-i/N","1/2 always"],
        "i/N",
        r"h_i=i/N.",
    ),
]

quiz_widgets = []
quiz_rows = []

for prompt, options, _, _ in quiz_data:
    dropdown = widgets.Dropdown(
        options=options,
        value="Choose...",
        layout=widgets.Layout(width="500px"),
    )

    quiz_widgets.append(dropdown)

    quiz_rows.append(widgets.HBox([
        widgets.HTML(
            f"<div style='width:720px'>{prompt}</div>"
        ),
        dropdown,
    ]))

grade_button = widgets.Button(description="Grade quiz")
quiz_output = widgets.Output()


def grade_quiz(_):
    with quiz_output:
        clear_output(wait=True)

        score = sum(
            widget.value == correct
            for widget, (_,_,correct,_) in zip(
                quiz_widgets,
                quiz_data,
            )
        )

        display(Markdown(
            f"### Score: {score}/{len(quiz_data)}"
        ))

        for i, (
            widget,
            (_,_,correct,explanation),
        ) in enumerate(zip(quiz_widgets,quiz_data),1):

            mark = "✓" if widget.value == correct else "✗"

            display(Markdown(
                f"**{mark} Question {i}:** correct answer = `{correct}`"
            ))

            display(Math(explanation))


grade_button.on_click(grade_quiz)

display(widgets.VBox(
    quiz_rows+[grade_button,quiz_output]
))


## 27. Automatic mathematical verification

The final code cell checks representative formulas from every major section.


In [ ]:
# Stochastic matrix.
assert is_stochastic(P_two)

# Chapman--Kolmogorov.
assert np.allclose(
    matrix_power(P_two,5),
    matrix_power(P_two,2) @ matrix_power(P_two,3),
)

# Path probability.
mu0 = np.array([1.0,0.0])
assert abs(
    path_probability(mu0,P_two,[0,0,1,1])
    - 0.8*0.2*0.7
) < 1e-12

# First-step equations.
h = hitting_probability(
    P_delay,
    A={2},
    B={0},
)
m = expected_hitting_time(
    P_delay,
    A={0,2},
)

assert abs(h[1]-0.6) < 1e-12
assert abs(m[1]-2) < 1e-12

# Stationary distribution.
pi = stationary_distribution(P_stationary)

assert np.allclose(pi,[0.6,0.4])
assert np.allclose(pi@P_stationary,pi)

# Kac two-state example.
assert abs(1/pi[0]-5/3) < 1e-12
assert abs(1/pi[1]-2.5) < 1e-12

# Detailed balance on path graph.
assert detailed_balance_error(P_path,pi_path) < 1e-12
assert np.allclose(pi_path@P_path,pi_path)

# Period calculations.
assert period_irreducible(P_alt) == 2
assert period_irreducible(P_cycle3) == 3
assert period_irreducible(P_graph) == 1

# Cyclic classes.
classes3 = cyclic_classes(P_cycle3)
assert classes3 == [[0],[1],[2]]

# Aperiodic convergence.
pi_ap = stationary_distribution(P_ap)
target_ap = np.ones((2,1)) @ pi_ap.reshape(1,-1)

assert np.max(
    np.abs(matrix_power(P_ap,50)-target_ap)
) < 1e-10

# Period-two phases and their average.
pi_p2 = stationary_distribution(P_period2)
target_p2 = np.ones((4,1)) @ pi_p2.reshape(1,-1)

L0 = matrix_power(P_period2,200)
L1 = matrix_power(P_period2,201)

assert np.max(
    np.abs((L0+L1)/2-target_p2)
) < 1e-12

# Three-cycle phase average.
C3 = (
    np.eye(3)
    + P_cycle3
    + matrix_power(P_cycle3,2)
)/3

target3 = np.ones((3,1)) @ pi3.reshape(1,-1)

assert np.allclose(C3,target3)

# Gambler's ruin fair case.
assert abs(
    gambler_ruin_probability(30,100,0.5)
    - 0.3
) < 1e-12

assert abs(
    gambler_ruin_duration(30,100,0.5)
    - 2100
) < 1e-12

# Biased gambler's ruin example.
biased = gambler_ruin_probability(4,10,0.6)

assert abs(
    biased
    -
    (
        1-(2/3)**4
    )/(
        1-(2/3)**10
    )
) < 1e-12

# Periodic scalar limit recovery.
assert abs(2*0.12-0.24) < 1e-12
assert abs(0.28/4-0.07) < 1e-12

# Random walk return probability for p=0.6.
assert abs(2*0.4-0.8) < 1e-12

show_result(
    "All Chapter 16 automatic checks passed",
    r"\mu^{(n)}=\mu^{(0)}P^n",
    r"h_i=\sum_jp_{ij}h_j",
    r"\mathbb E_i[T_i^+]=1/\pi_i",
    r"\pi_i p_{ij}=\pi_jp_{ji}\Longrightarrow\pi P=\pi",
    r"P^{nd+a}\longrightarrow L_a",
    r"\frac1d\sum_{a=0}^{d-1}L_a=\mathbf1\pi",
    note=(
        "Transition, first-step, stationarity, Kac, reversibility, period, "
        "periodic-limit and gambler's-ruin checks all passed."
    ),
)


## 28. Chapter map

| Chapter concept | Computational representation |
|---|---|
| stochastic process | simulated state trajectory |
| Markov property | conditional one-step law |
| natural filtration | information up to time $n$ |
| Markov operator | matrix applied to a payoff vector |
| transition matrix | stochastic-matrix validator |
| $n$-step probabilities | matrix powers |
| path probability | product of transition probabilities |
| Chapman--Kolmogorov | matrix multiplication check |
| hitting time | target-set first passage |
| stopping time | observable-by-time-$n$ event |
| strong Markov property | restart at a stopping time |
| first-step analysis | finite linear systems |
| communication | graph reachability |
| irreducibility | mutual reachability of every pair |
| recurrence/transience | return probability and return series |
| finite positive recurrence | finite mean return times |
| stationary distribution | linear system $\pi P=\pi$ |
| Kac formula | reciprocal stationary mass |
| reversibility | detailed-balance error |
| graph random walk | degree-proportional stationary law |
| period | graph gcd algorithm |
| cyclic classes | residue classes of path lengths |
| equal cyclic mass | $\pi(C_a)=1/d$ |
| aperiodic convergence | $P^n\to\mathbf1\pi$ |
| periodic phase limits | $L_a$ matrices |
| countable-state caveat | multiplication-order warning |
| Cesàro limit | average of matrix powers |
| phase-average identity | $\frac1d\sum_aL_a=\mathbf1\pi$ |
| sample-path ergodic theorem | occupation frequencies |
| period-two example | alternating phase limits |
| period-three example | $I,P,P^2$ |
| gambler's ruin | probability and duration formulas |
| random walk | recurrent/transient classification |
| Python laboratory | stationary law, period, phases, time averages |
| AI Audit | theorem-hypothesis checks |

The chapter's central long-run picture is:

$$
\boxed{
\text{aperiodic: }P^n\to\mathbf1\pi,
}
$$

while

$$
\boxed{
\text{periodic: }P^n\text{ oscillates by phase, but }
\frac1N\sum_{n=0}^{N-1}P^n\to\mathbf1\pi.
}
$$

At the sample-path level, finite irreducibility is enough for long-run occupation frequencies to converge almost surely to $\pi$.
